# Model Comparison — Analisis Komparatif Semua Model

**Notebook:** `06_model_comparison.ipynb`  
**Peneliti:** Khoeru Roziqin  
**Tanggal:** 01-04-26  

---

## Deskripsi

Notebook ini menyajikan **analisis perbandingan komprehensif** seluruh model yang telah 
dilatih dan dievaluasi dalam penelitian ini:

| # | Model | Notebook | Kondisi |
|---|---|---|---|
| 1 | **IndoBERT+CNN Dual-Path** (Proposed) | nb03 | S2 — Random Undersampling |
| 2 | **IndoBERT-only** (DL Baseline) | nb04 | S2 — Random Undersampling |
| 3 | **SVM** (ML Baseline) | nb05 | S1 — class_weight=balanced |
| 4 | **Logistic Regression** (ML Baseline) | nb05 | S1 — class_weight=balanced |
| 5 | **Random Forest** (ML Baseline) | nb05 | S1 — class_weight=balanced |

## Hasil Akhir

```
Model                       F1-Macro  Accuracy  K-Fold F1     Std
IndoBERT+CNN (Proposed)      0.8547    0.8570    0.8461    ±0.0155
IndoBERT-only (DL Baseline)  0.8630    0.8668    0.8429    ±0.0188
SVM (ML Baseline)            0.8267    0.8322    0.8023    ±0.0107
Logistic Regression          0.8251    0.8284    0.8029    ±0.0113
Random Forest                0.7966    0.8006    0.7706    ±0.0115
```

## Test Set

Semua model dievaluasi pada **test set yang identik** (1,329 tweet) menggunakan 
`test_set_indices.npy` dari notebook 03.

## 0. Setup

Notebook ini tidak membutuhkan GPU.

In [ ]:
import sys, os
print(f'Python : {sys.version}')
print(f'CWD    : {os.getcwd()}')
print('✅ Setup selesai. (CPU — tidak membutuhkan GPU)')

## 1. Import Library

In [ ]:
import warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from math import pi
warnings.filterwarnings('ignore')

print('✅ Library berhasil dimuat.')

## 2. Konfigurasi & Nilai Hasil Final

Seluruh nilai metrik final dari notebook 03, 04, dan 05 didefinisikan terpusat. 
Nilai ini juga diverifikasi terhadap `bert_only_summary.csv` dan `ml_baseline_summary.csv` 
yang di-restore dari dataset.

In [ ]:
# ── PATH ──────────────────────────────────────────────────────────────────────
OUTPUT_DIR    = '/kaggle/working/comparison_results'
OUTPUTS_INPUT = '/kaggle/input/datasets/khoeruroziqin/final-training-dual-path-outputs'
RESTORE_DIR   = '/kaggle/working/restored'
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(RESTORE_DIR, exist_ok=True)

# ── Restore summary CSV dari dataset ─────────────────────────────────────────
_files = ['bert_only_summary.csv', 'ml_baseline_summary.csv']
print('Restoring files dari final-training-dual-path-outputs...')
for fname in _files:
    src = f'{OUTPUTS_INPUT}/{fname}'
    dst = f'{RESTORE_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  ✔ {fname}')
    else:
        print(f'  ⚠️  Tidak ditemukan: {fname}')
print('Done.')

# ── KONSTANTA ─────────────────────────────────────────────────────────────────
LABEL_NAMES = ['positive', 'negative', 'neutral']
COLORS      = {
    'proposed' : '#534AB7',
    'bert_only': '#E24B4A',
    'svm'      : '#1D9E75',
    'lr'       : '#EF9F27',
    'rf'       : '#AFA9EC',
}

# ── NILAI METRIK FINAL (dari notebook 03, 04, 05) ────────────────────────────
# Sumber: log dan output masing-masing notebook

# Proposed — IndoBERT+CNN Dual-Path (nb03, kondisi S2)
PROPOSED = {
    'model'      : 'IndoBERT+CNN\n(Proposed)',
    'model_full' : 'IndoBERT+CNN Dual-Path (Proposed)',
    'condition'  : 'S2 (Undersampling)',
    'notebook'   : 'nb03',
    'f1_macro'   : 0.8547,
    'f1_weighted': 0.8587,
    'accuracy'   : 0.8570,
    'precision'  : 0.8551,
    'recall'     : 0.8585,
    'f1_positive': 0.8891,
    'f1_negative': 0.8717,
    'f1_neutral' : 0.8034,
    'kfold_mean' : 0.8461,
    'kfold_std'  : 0.0155,
    'color'      : COLORS['proposed'],
}

# DL Baseline — IndoBERT-only (nb04, kondisi S2)
BERT_ONLY = {
    'model'      : 'IndoBERT-only\n(DL Baseline)',
    'model_full' : 'IndoBERT-only (DL Baseline)',
    'condition'  : 'S2 (Undersampling)',
    'notebook'   : 'nb04',
    'f1_macro'   : 0.8630,
    'f1_weighted': 0.8674,
    'accuracy'   : 0.8668,
    'precision'  : 0.8621,
    'recall'     : 0.8645,
    'f1_positive': 0.9017,
    'f1_negative': 0.8789,
    'f1_neutral' : 0.8086,
    'kfold_mean' : 0.8429,
    'kfold_std'  : 0.0188,
    'color'      : COLORS['bert_only'],
}

# ML Baseline — SVM (nb05, kondisi S1)
SVM = {
    'model'      : 'SVM\n(ML Baseline)',
    'model_full' : 'SVM (ML Baseline)',
    'condition'  : 'S1 (class_weight=balanced)',
    'notebook'   : 'nb05',
    'f1_macro'   : 0.8267,
    'f1_weighted': 0.8313,
    'accuracy'   : 0.8322,
    'precision'  : 0.8280,
    'recall'     : 0.8269,
    'f1_positive': 0.8658,
    'f1_negative': 0.8482,
    'f1_neutral' : 0.7661,
    'kfold_mean' : 0.8023,
    'kfold_std'  : 0.0107,
    'color'      : COLORS['svm'],
}

# ML Baseline — Logistic Regression (nb05, kondisi S1)
LR = {
    'model'      : 'Logistic Reg.\n(ML Baseline)',
    'model_full' : 'Logistic Regression (ML Baseline)',
    'condition'  : 'S1 (class_weight=balanced)',
    'notebook'   : 'nb05',
    'f1_macro'   : 0.8251,
    'f1_weighted': 0.8287,
    'accuracy'   : 0.8284,
    'precision'  : 0.8251,
    'recall'     : 0.8284,
    'f1_positive': 0.8560,
    'f1_negative': 0.8416,
    'f1_neutral' : 0.7778,
    'kfold_mean' : 0.8029,
    'kfold_std'  : 0.0113,
    'color'      : COLORS['lr'],
}

# ML Baseline — Random Forest (nb05, kondisi S1)
RF = {
    'model'      : 'Random Forest\n(ML Baseline)',
    'model_full' : 'Random Forest (ML Baseline)',
    'condition'  : 'S1 (class_weight=balanced)',
    'notebook'   : 'nb05',
    'f1_macro'   : 0.7966,
    'f1_weighted': 0.8005,
    'accuracy'   : 0.8006,
    'precision'  : 0.7975,
    'recall'     : 0.7995,
    'f1_positive': 0.8299,
    'f1_negative': 0.8143,
    'f1_neutral' : 0.7457,
    'kfold_mean' : 0.7706,
    'kfold_std'  : 0.0115,
    'color'      : COLORS['rf'],
}

ALL_MODELS = [PROPOSED, BERT_ONLY, SVM, LR, RF]

print('✅ Konfigurasi berhasil dimuat.')
print(f'   Jumlah model yang dibandingkan: {len(ALL_MODELS)}')
print(f'   Output dir: {OUTPUT_DIR}')

## 3. Tabel Ringkasan Semua Model

Tabel metrik lengkap dari seluruh model ditampilkan dan disimpan ke CSV 
sebagai dokumentasi resmi hasil penelitian.

In [ ]:
# ── Bangun DataFrame ringkasan ────────────────────────────────────────────────
rows = []
for m in ALL_MODELS:
    rows.append({
        'Model'            : m['model_full'],
        'Notebook'         : m['notebook'],
        'Kondisi'          : m['condition'],
        'F1-Macro'         : m['f1_macro'],
        'F1-Weighted'      : m['f1_weighted'],
        'Accuracy'         : m['accuracy'],
        'Precision'        : m['precision'],
        'Recall'           : m['recall'],
        'F1-Positive'      : m['f1_positive'],
        'F1-Negative'      : m['f1_negative'],
        'F1-Neutral'       : m['f1_neutral'],
        'KFold-F1-Mean'    : m['kfold_mean'],
        'KFold-F1-Std'     : m['kfold_std'],
    })

df_summary = pd.DataFrame(rows)
df_summary.to_csv(f'{OUTPUT_DIR}/model_comparison_summary.csv',
                  index=False, encoding='utf-8-sig')

# ── Cetak tabel ringkasan ─────────────────────────────────────────────────────
print('=' * 75)
print(' RINGKASAN PERBANDINGAN SEMUA MODEL — TEST SET')
print('=' * 75)
print(f' {"Model":<35} {"F1-Macro":>9} {"Accuracy":>9} {"KFold F1":>9} {"Std":>7}')
print('─' * 75)
for m in ALL_MODELS:
    marker = ' ←' if m == BERT_ONLY else ('  ★' if m == PROPOSED else '')
    print(f' {m["model_full"]:<35} '
          f'{m["f1_macro"]:>9.4f} '
          f'{m["accuracy"]:>9.4f} '
          f'{m["kfold_mean"]:>9.4f} '
          f'{m["kfold_std"]:>7.4f}{marker}')
print('─' * 75)
best = max(ALL_MODELS, key=lambda x: x['f1_macro'])
print(f' Best overall: {best["model_full"]} (F1-Macro={best["f1_macro"]:.4f})')
print('=' * 75)
print()
print(' Per-class F1:')
print(f' {"Model":<35} {"Positive":>10} {"Negative":>10} {"Neutral":>10}')
print('─' * 70)
for m in ALL_MODELS:
    print(f' {m["model_full"]:<35} '
          f'{m["f1_positive"]:>10.4f} '
          f'{m["f1_negative"]:>10.4f} '
          f'{m["f1_neutral"]:>10.4f}')

print(f'\n✔ model_comparison_summary.csv tersimpan')

## 4. Visualisasi Perbandingan

Lima chart komparatif dihasilkan untuk keperluan laporan:

1. **F1-Macro comparison** — bar chart semua model (test set + K-Fold)
2. **Metrik lengkap** — grouped bar chart (F1, Accuracy, Precision, Recall)
3. **Per-class F1** — grouped bar chart per kelas sentimen
4. **K-Fold stability** — error bar plot mean ± std
5. **Radar chart** — profil metrik multi-dimensi

In [ ]:
model_labels  = [m['model'] for m in ALL_MODELS]
model_colors  = [m['color'] for m in ALL_MODELS]
model_labels_full = [m['model_full'] for m in ALL_MODELS]

# ── CHART 1: F1-Macro Test vs K-Fold ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
x     = np.arange(len(ALL_MODELS))
width = 0.35

bars1 = ax.bar(x - width/2,
               [m['f1_macro']   for m in ALL_MODELS],
               width, label='Test Set F1-Macro',
               color=model_colors, edgecolor='none', alpha=0.9)
bars2 = ax.bar(x + width/2,
               [m['kfold_mean'] for m in ALL_MODELS],
               width, label='K-Fold F1-Macro (mean)',
               color=model_colors, edgecolor='none', alpha=0.45)
ax.errorbar(x + width/2,
            [m['kfold_mean'] for m in ALL_MODELS],
            yerr=[m['kfold_std'] for m in ALL_MODELS],
            fmt='none', ecolor='#444441', capsize=5, linewidth=1.5)

# Annotasi nilai
for bar, m in zip(bars1, ALL_MODELS):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.003,
            f'{m["f1_macro"]:.4f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold')
for bar, m in zip(bars2, ALL_MODELS):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.003,
            f'{m["kfold_mean"]:.4f}',
            ha='center', va='bottom', fontsize=7, color='#555552')

ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=9)
ax.set_ylabel('F1-Macro', fontsize=11)
ax.set_title('Perbandingan F1-Macro — Test Set vs K-Fold Cross Validation\n'
             '(bar penuh=test set, bar transparan=K-Fold mean ± std)',
             fontsize=11, pad=12)
ax.set_ylim(0.72, 0.92)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.legend(fontsize=10)
ax.axhline(y=PROPOSED['f1_macro'], color=COLORS['proposed'],
           linestyle=':', linewidth=1.2, alpha=0.6,
           label=f'Proposed F1={PROPOSED["f1_macro"]}')
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_f1macro.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ comparison_f1macro.png tersimpan')

# ── CHART 2: Metrik Lengkap (grouped bar) ────────────────────────────────────
metrics_list  = ['f1_macro','f1_weighted','accuracy','precision','recall']
metrics_label = ['F1-Macro','F1-Weighted','Accuracy','Precision','Recall']
x2    = np.arange(len(metrics_list))
w2    = 0.15
offsets = np.linspace(-(len(ALL_MODELS)-1)/2, (len(ALL_MODELS)-1)/2, len(ALL_MODELS)) * w2

fig, ax = plt.subplots(figsize=(14, 6))
for i, (m, color, offset) in enumerate(zip(ALL_MODELS, model_colors, offsets)):
    vals = [m[k] for k in metrics_list]
    bars = ax.bar(x2 + offset, vals, w2,
                  label=m['model_full'], color=color,
                  edgecolor='none', alpha=0.9)

ax.set_xticks(x2)
ax.set_xticklabels(metrics_label, fontsize=10)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Perbandingan Metrik Lengkap — Semua Model (Test Set)', fontsize=12, pad=12)
ax.set_ylim(0.75, 0.95)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.legend(fontsize=8, loc='lower right')
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_metrics_full.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ comparison_metrics_full.png tersimpan')

# ── CHART 3: Per-class F1 (grouped bar) ──────────────────────────────────────
x3 = np.arange(len(LABEL_NAMES))
w3 = 0.15

fig, ax = plt.subplots(figsize=(12, 6))
for i, (m, color, offset) in enumerate(zip(ALL_MODELS, model_colors, offsets)):
    vals = [m['f1_positive'], m['f1_negative'], m['f1_neutral']]
    ax.bar(x3 + offset, vals, w3,
           label=m['model_full'], color=color,
           edgecolor='none', alpha=0.9)

ax.set_xticks(x3)
ax.set_xticklabels(['Positive', 'Negative', 'Neutral'], fontsize=11)
ax.set_ylabel('F1-Score per Kelas', fontsize=11)
ax.set_title('Perbandingan Per-class F1 — Semua Model (Test Set)', fontsize=12, pad=12)
ax.set_ylim(0.70, 0.95)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.legend(fontsize=8, loc='lower right')
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_perclass_f1.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ comparison_perclass_f1.png tersimpan')

# ── CHART 4: K-Fold Stability (mean ± std error bar) ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
for i, (m, color) in enumerate(zip(ALL_MODELS, model_colors)):
    ax.errorbar(i, m['kfold_mean'], yerr=m['kfold_std'],
                fmt='o', color=color, markersize=10,
                capsize=8, capthick=2, linewidth=2,
                label=m['model_full'])
    ax.text(i, m['kfold_mean']+m['kfold_std']+0.004,
            f'{m["kfold_mean"]:.4f}\n±{m["kfold_std"]:.4f}',
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(range(len(ALL_MODELS)))
ax.set_xticklabels(model_labels, fontsize=9)
ax.set_ylabel('K-Fold F1-Macro (Mean ± Std)', fontsize=11)
ax.set_title('Stabilitas K-Fold Cross Validation — Semua Model', fontsize=12, pad=12)
ax.set_ylim(0.72, 0.90)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.grid(axis='y', alpha=0.25)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_kfold_stability.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ comparison_kfold_stability.png tersimpan')

# ── CHART 5: Radar Chart ──────────────────────────────────────────────────────
radar_metrics = ['f1_macro','accuracy','f1_positive','f1_negative','f1_neutral','kfold_mean']
radar_labels  = ['F1-Macro','Accuracy','F1-Pos','F1-Neg','F1-Neu','KFold F1']
N = len(radar_metrics)
angles = [n/float(N)*2*pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for m, color in zip(ALL_MODELS, model_colors):
    vals   = [m[k] for k in radar_metrics]
    vals  += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=1.8,
            color=color, label=m['model_full'], alpha=0.85)
    ax.fill(angles, vals, alpha=0.07, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)
ax.set_ylim(0.72, 0.95)
ax.set_yticks([0.75, 0.80, 0.85, 0.90])
ax.set_yticklabels(['0.75','0.80','0.85','0.90'], fontsize=7)
ax.set_title('Radar Chart — Profil Performa Semua Model', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_radar.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ comparison_radar.png tersimpan')

print('\n✅ Semua visualisasi perbandingan tersimpan.')

## 5. Heatmap Metrik Lengkap

Heatmap menampilkan semua metrik semua model secara bersamaan 
untuk mempermudah identifikasi pola kekuatan dan kelemahan tiap model.

In [ ]:
# Bangun matrix untuk heatmap
hm_metrics = [
    ('f1_macro',   'F1-Macro'),
    ('accuracy',   'Accuracy'),
    ('precision',  'Precision'),
    ('recall',     'Recall'),
    ('f1_positive','F1-Positive'),
    ('f1_negative','F1-Negative'),
    ('f1_neutral', 'F1-Neutral'),
    ('kfold_mean', 'KFold F1'),
    ('kfold_std',  'KFold Std'),
]

hm_data  = pd.DataFrame(
    {m['model_full']: {lbl: m[key] for key, lbl in hm_metrics}
     for m in ALL_MODELS}
).T

fig, ax = plt.subplots(figsize=(13, 6))
hm = sns.heatmap(
    hm_data, annot=True, fmt='.4f',
    cmap='YlOrRd', linewidths=0.5,
    vmin=0.70, vmax=0.92,
    ax=ax, cbar_kws={'label': 'Score'}
)
ax.set_title('Heatmap Metrik Lengkap — Semua Model', fontsize=12, pad=12)
ax.set_xlabel('Metrik', fontsize=11)
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ comparison_heatmap.png tersimpan')

## 6. Analisis Statistik

Analisis selisih performa antar model dan diskusi tentang pola yang ditemukan.

In [ ]:
print('=' * 70)
print(' ANALISIS KOMPARATIF')
print('=' * 70)

# ── DL vs ML Gap ──────────────────────────────────────────────────────────────
best_ml_f1   = max(SVM['f1_macro'], LR['f1_macro'], RF['f1_macro'])
best_ml_name = 'SVM' if SVM['f1_macro'] == best_ml_f1 else \
               'LR'  if LR['f1_macro']  == best_ml_f1 else 'RF'
dl_vs_best_ml_proposed  = PROPOSED['f1_macro']  - best_ml_f1
dl_vs_best_ml_bertonly  = BERT_ONLY['f1_macro'] - best_ml_f1

print(f'\n 1. Deep Learning vs Machine Learning:')
print(f'    Best ML ({best_ml_name})          : F1 = {best_ml_f1:.4f}')
print(f'    Proposed (IndoBERT+CNN)   : F1 = {PROPOSED["f1_macro"]:.4f} '
      f'(+{dl_vs_best_ml_proposed:.4f} vs best ML)')
print(f'    BERT-only                 : F1 = {BERT_ONLY["f1_macro"]:.4f} '
      f'(+{dl_vs_best_ml_bertonly:.4f} vs best ML)')

# ── Proposed vs BERT-only ─────────────────────────────────────────────────────
delta_proposed_vs_bert = PROPOSED['f1_macro'] - BERT_ONLY['f1_macro']
print(f'\n 2. Proposed (IndoBERT+CNN) vs BERT-only Baseline:')
print(f'    Delta F1-Macro  : {delta_proposed_vs_bert:+.4f} '
      f'({"proposed lebih baik" if delta_proposed_vs_bert > 0 else "BERT-only lebih baik"})')
print(f'    Delta Accuracy  : {PROPOSED["accuracy"]-BERT_ONLY["accuracy"]:+.4f}')
print(f'    K-Fold overlap  : proposed={PROPOSED["kfold_mean"]:.4f}±{PROPOSED["kfold_std"]:.4f} '
      f'vs bert={BERT_ONLY["kfold_mean"]:.4f}±{BERT_ONLY["kfold_std"]:.4f}')
print(f'    → Perbedaan test set kecil (0.0083), K-Fold std saling overlap')
print(f'      → Tidak signifikan secara statistik; kedua DL model sebanding')

# ── Kelas Neutral ─────────────────────────────────────────────────────────────
print(f'\n 3. Analisis Per-class F1:')
print(f'    Kelas Neutral — paling sulit di semua model:')
for m in ALL_MODELS:
    print(f'      {m["model_full"]:<35}: {m["f1_neutral"]:.4f}')
print(f'    Kelas Positive — paling mudah di semua model:')
for m in ALL_MODELS:
    print(f'      {m["model_full"]:<35}: {m["f1_positive"]:.4f}')

# ── K-Fold Stability ──────────────────────────────────────────────────────────
print(f'\n 4. Stabilitas K-Fold (std — semakin kecil semakin stabil):')
sorted_by_std = sorted(ALL_MODELS, key=lambda x: x['kfold_std'])
for m in sorted_by_std:
    print(f'    {m["model_full"]:<35}: std={m["kfold_std"]:.4f}')

# ── Ranking Final ─────────────────────────────────────────────────────────────
print(f'\n 5. Ranking Final (berdasarkan Test F1-Macro):')
ranked = sorted(ALL_MODELS, key=lambda x: x['f1_macro'], reverse=True)
for rank, m in enumerate(ranked, 1):
    marker = ' ← BEST' if rank == 1 else (' ← Proposed' if m == PROPOSED else '')
    print(f'    [{rank}] {m["model_full"]:<35}: {m["f1_macro"]:.4f}{marker}')
print(f'\n  ✔ analisis_komparatif selesai.')

## 7. Verifikasi dengan CSV dari Dataset

Nilai hardcode di Cell Config diverifikasi terhadap CSV yang di-restore 
dari dataset untuk memastikan konsistensi data.

In [ ]:
print('Verifikasi nilai hardcode vs CSV yang di-restore...')
errors = []

# Verifikasi bert_only_summary.csv
_bo_path = f'{RESTORE_DIR}/bert_only_summary.csv'
if os.path.exists(_bo_path):
    df_bo = pd.read_csv(_bo_path)
    bo_f1 = round(float(df_bo['test_f1_macro'].iloc[0]), 4)
    if abs(bo_f1 - BERT_ONLY['f1_macro']) > 1e-4:
        errors.append(f'  ❌ BERT-only F1: CSV={bo_f1} vs config={BERT_ONLY["f1_macro"]}')
    else:
        print(f'  ✅ BERT-only F1-Macro: {bo_f1} (match)')
else:
    print(f'  ⚠️  bert_only_summary.csv tidak ditemukan — skip verifikasi')

# Verifikasi ml_baseline_summary.csv
_ml_path = f'{RESTORE_DIR}/ml_baseline_summary.csv'
if os.path.exists(_ml_path):
    df_ml = pd.read_csv(_ml_path)
    for m_name, m_dict in [('SVM', SVM),
                            ('LogisticRegression', LR),
                            ('RandomForest', RF)]:
        row = df_ml[df_ml['model'] == m_name]
        if len(row) > 0:
            csv_f1 = round(float(row['test_f1_macro'].iloc[0]), 4)
            if abs(csv_f1 - m_dict['f1_macro']) > 1e-4:
                errors.append(
                    f'  ❌ {m_name} F1: CSV={csv_f1} vs config={m_dict["f1_macro"]}')
            else:
                print(f'  ✅ {m_name} F1-Macro: {csv_f1} (match)')
else:
    print(f'  ⚠️  ml_baseline_summary.csv tidak ditemukan — skip verifikasi')

if errors:
    print('\n❌ Ada ketidaksesuaian nilai:')
    for e in errors: print(e)
    print('  → Perbarui nilai di Cell Config!')
else:
    print('\n✅ Semua nilai terverifikasi — konsisten dengan CSV.')

## 8. Ringkasan Output

In [ ]:
print('=' * 65)
print(' OUTPUT NOTEBOOK 06 — MODEL COMPARISON')
print('=' * 65)
outputs = [
    'model_comparison_summary.csv',
    'comparison_f1macro.png',
    'comparison_metrics_full.png',
    'comparison_perclass_f1.png',
    'comparison_kfold_stability.png',
    'comparison_radar.png',
    'comparison_heatmap.png',
]
descs = [
    'Tabel ringkasan semua metrik semua model (CSV)',
    'F1-Macro test set vs K-Fold comparison',
    'Grouped bar — metrik lengkap semua model',
    'Per-class F1 semua model',
    'Stabilitas K-Fold — error bar mean ± std',
    'Radar chart — profil multi-dimensi',
    'Heatmap — semua metrik semua model',
]
for fname, desc in zip(outputs, descs):
    exists = '✔' if os.path.exists(f'{OUTPUT_DIR}/{fname}') else '○'
    print(f'  {exists} {fname:<45} {desc}')

print('\n Pipeline penelitian selesai.')
print('─' * 65)
print(' Ringkasan final:')
for rank, m in enumerate(
    sorted(ALL_MODELS, key=lambda x: x['f1_macro'], reverse=True), 1):
    print(f'   [{rank}] {m["model_full"]:<35}: F1={m["f1_macro"]:.4f} '
          f'| Acc={m["accuracy"]:.4f}')
print('─' * 65)
print(' Seluruh hasil siap untuk penulisan Bab 4 (Hasil dan Pembahasan).')

---

## Catatan Metodologis untuk Laporan

### Konsistensi Evaluasi
Seluruh model dievaluasi pada test set yang identik (1,329 tweet, 20% dari total data) 
menggunakan indeks yang di-generate sekali di notebook 03 dan digunakan bersama 
oleh notebook 04 dan 05. Hal ini memastikan perbandingan yang *fair* dan valid.

### IndoBERT-only Lebih Tinggi dari Proposed
IndoBERT-only (F1=0.8630) sedikit lebih tinggi dari proposed model IndoBERT+CNN (F1=0.8547) 
dengan selisih 0.0083. Beberapa penjelasan yang didukung literatur:
- **Teks pendek**: Tweet rata-rata pendek sehingga representasi `[CLS]` BERT sudah sangat 
  kaya tanpa membutuhkan CNN untuk mengekstrak pola n-gram lokal (Kim, 2014).
- **Parameter lebih sedikit**: IndoBERT-only lebih sederhana, lebih mudah dioptimasi 
  dengan data 5,313 sampel training.
- **Tidak signifikan**: Selisih 0.0083 berada dalam rentang overlap std K-Fold kedua model, 
  sehingga tidak dapat diklaim berbeda secara statistik.

### DL vs ML Gap
Semua model DL (IndoBERT+CNN dan IndoBERT-only) mengungguli semua model ML 
(SVM, LR, RF) dengan margin yang konsisten (~0.03-0.06 F1-Macro). 
Hal ini mengkonfirmasi keunggulan representasi kontekstual berbasis Transformer 
untuk analisis sentimen teks berbahasa Indonesia.

### Kelas Neutral — Tantangan Universal
Kelas Neutral memiliki F1 terendah di semua model, mencerminkan tantangan inheren 
dalam membedakan sentimen netral dari sentimen positif/negatif pada teks Twitter.
Pola ini konsisten dengan temuan pada penelitian analisis sentimen berbahasa Indonesia 
lainnya (Koto et al., 2020).

**Referensi:**  
Kim, Y. (2014). Convolutional neural networks for sentence classification. *EMNLP 2014*.  
Koto, F., et al. (2020). IndoLEM and IndoBERT. *COLING 2020*.